In [1]:
!pip install -q pymupdf pillow

import fitz
from PIL import Image

# === INPUT FILES    (your 4 LLM maps) ===
p_gpt    = "map_pi_gpt.pdf"
p_claude = "map_pi_claude.pdf"
p_gemini = "map_pi_gemini.pdf"
p_llama  = "map_pi_llama.pdf"

# === OUTPUT FILE ===
panel_B = "Fig_maps_2x2_VECTOR.pdf"

PAGE_W = 1800
MARGIN = 50
GAP = 40

CELL_W = (PAGE_W - 2*MARGIN - GAP) / 2

def scaled_h(pdf,w):
    d=fitz.open(pdf);r=d[0].rect
    h = (w/r.width)*r.height
    d.close()
    return h

h_row1 = max(scaled_h(p_gpt,CELL_W),
             scaled_h(p_claude,CELL_W))
h_row2 = max(scaled_h(p_gemini,CELL_W),
             scaled_h(p_llama, CELL_W))

PAGE_H = MARGIN + h_row1 + GAP + h_row2 + MARGIN

doc = fitz.open()
page = doc.new_page(width=PAGE_W,height=PAGE_H)

def place(p,x,y,w):
    d=fitz.open(p)
    r=d[0].rect
    s=w/r.width
    h=r.height*s
    page.show_pdf_page(fitz.Rect(x,y,x+w,y+h), d,0)
    d.close()
    return h

y=MARGIN
place(p_gpt,    MARGIN,               y, CELL_W)
place(p_claude, MARGIN+CELL_W+GAP,    y, CELL_W)
y+=h_row1+GAP
place(p_gemini, MARGIN,               y, CELL_W)
place(p_llama,  MARGIN+CELL_W+GAP,    y, CELL_W)

doc.save(panel_B)
doc.close()
print("Panel B generated:",panel_B)



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Panel B generated: Fig_maps_2x2_VECTOR.pdf


In [10]:
!pip install -q pymupdf pillow

import fitz
from PIL import Image

# ========= INPUT FILES =========
# individual maps for B panel
p_gpt    = "map_pi_gpt.pdf"
p_claude = "map_pi_claude.pdf"
p_gemini = "map_pi_gemini.pdf"
p_llama  = "map_pi_llama.pdf"

# A, C, D panels
panel_A_src = "map_pi_ground_truth.pdf"
panel_C     = "pi_stage8_scatter_actual_vs_predicted.pdf"
panel_D     = "ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf"

# ========= OUTPUT FILES =========
panel_B_path = "Fig_maps_2x2_VECTOR.pdf"
out_pdf = "Fig_ABCD_2x2_BALANCED_VECTOR.pdf"
out_png = "Fig_ABCD_2x2_BALANCED_600dpi.png"


# ========= HELPER FUNCTIONS =========
def scaled_h(pdf_path, target_w):
    """Return height of first page when scaled to width target_w."""
    d = fitz.open(pdf_path)
    r = d[0].rect
    s = target_w / r.width
    h = r.height * s
    d.close()
    return h

def place_pdf(page, pdf_path, x, y, target_w):
    """Place first page of pdf_path on page at (x,y) scaled to width target_w."""
    d = fitz.open(pdf_path)
    r = d[0].rect
    s = target_w / r.width
    h = r.height * s
    dest = fitz.Rect(x, y, x + target_w, y + h)
    page.show_pdf_page(dest, d, 0)
    d.close()
    return h


# ========= STEP 1: BUILD PANEL B (2×2 LLM MAPS) =========
PAGE_W_B = 1800
MARGIN_B = 50
GAP_B = 40
CELL_W_B = (PAGE_W_B - 2*MARGIN_B - GAP_B) / 2

h_row1_B = max(scaled_h(p_gpt, CELL_W_B),
               scaled_h(p_claude, CELL_W_B))
h_row2_B = max(scaled_h(p_gemini, CELL_W_B),
               scaled_h(p_llama,  CELL_W_B))

PAGE_H_B = MARGIN_B + h_row1_B + GAP_B + h_row2_B + MARGIN_B

docB = fitz.open()
pageB = docB.new_page(width=PAGE_W_B, height=PAGE_H_B)

y = MARGIN_B
place_pdf(pageB, p_gpt,    MARGIN_B,                y, CELL_W_B)
place_pdf(pageB, p_claude, MARGIN_B + CELL_W_B + GAP_B, y, CELL_W_B)
y += h_row1_B + GAP_B
place_pdf(pageB, p_gemini, MARGIN_B,                y, CELL_W_B)
place_pdf(pageB, p_llama,  MARGIN_B + CELL_W_B + GAP_B, y, CELL_W_B)

docB.save(panel_B_path)
docB.close()
print("Panel B created:", panel_B_path)


# ========= STEP 2: BUILD 2×2 MASTER (A,B,C,D) =========
PAGE_W = 1800
MARGIN = 50
GAP_H = 40   # horizontal gap between left/right
GAP_V = 50   # vertical gap between top/bottom

inner_full = PAGE_W - 2*MARGIN

# --- top row: A and B share width 50/50 ---
top_cell_w = (inner_full - GAP_H) / 2

panel_A = panel_A_src     # (use GT directly)
panel_B = panel_B_path    # (LLM 2×2 we just built)

hA = scaled_h(panel_A, top_cell_w)
hB = scaled_h(panel_B, top_cell_w)
row1_h = max(hA, hB)

# --- bottom row: C and D with different widths (C narrower, D wider) ---
C_frac = 0.40             # C uses 40% of inner width
D_frac = 0.60             # D uses 60% of inner width

C_w = C_frac * inner_full
D_w = D_frac * inner_full - GAP_H  # subtract gap explicitly

hC = scaled_h(panel_C, C_w)
hD = scaled_h(panel_D, D_w)
row2_h = max(hC, hD)

PAGE_H = MARGIN + row1_h + GAP_V + row2_h + MARGIN

doc = fitz.open()
page = doc.new_page(width=PAGE_W, height=PAGE_H)

# --- coordinates ---
x_left_top  = MARGIN
x_right_top = MARGIN + top_cell_w + GAP_H

y_top    = MARGIN

# shift the *bottom row* down a little
y_bottom = MARGIN + row1_h + GAP_V 

x_left_bottom  = MARGIN
x_right_bottom = MARGIN + C_w + GAP_H

# place panels
place_pdf(page, panel_A, x_left_top,  y_top,    top_cell_w)
place_pdf(page, panel_B, x_right_top, y_top,    top_cell_w)

place_pdf(page, panel_C, x_left_bottom,  y_bottom, C_w)
place_pdf(page, panel_D, x_right_bottom, y_bottom + 130, D_w)  # shift ONLY D


# ========= PANEL LABELS =========
def add_label(letter, x, y):
    page.insert_text(
        (x, y),
        letter,
        fontsize=26,
        fontname="helv",
        color=(0, 0, 0),
    )

LABEL_Y_OFFSET = 20  # distance above panel

add_label("A", x_left_top,      y_top    - LABEL_Y_OFFSET)
add_label("B", x_right_top,     y_top    - LABEL_Y_OFFSET)
add_label("C", x_left_bottom,   y_bottom - LABEL_Y_OFFSET)
add_label("D", x_right_bottom,  y_bottom - LABEL_Y_OFFSET)

# ========= SAVE MASTER VECTOR PDF =========
doc.save(out_pdf)
doc.close()
print("Master PDF saved:", out_pdf)

# ========= EXPORT 600-DPI PNG =========
pdf = fitz.open(out_pdf)
pix = pdf[0].get_pixmap(dpi=600)
img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
img.save(out_png)
pdf.close()
print("PNG saved:", out_png)



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Panel B created: Fig_maps_2x2_VECTOR.pdf
Master PDF saved: Fig_ABCD_2x2_BALANCED_VECTOR.pdf
PNG saved: Fig_ABCD_2x2_BALANCED_600dpi.png


In [3]:
!pip install -q pymupdf pillow

import fitz
from PIL import Image

# ================== INPUTS ==================
TOP_PANEL    = "original_mae_heatmap.pdf"      # TOP
BOTTOM_PANEL = "cf_original_4panel.pdf"        # BOTTOM

# ================== OUTPUT ==================
OUT_PDF = "Ablation_COMPARE_ORIGINAL_vs_CF_2x1.pdf"
OUT_PNG = "Ablation_COMPARE_ORIGINAL_vs_CF_2x1_600dpi.png"


# ================== helper functions ==================
def scaled_height(pdf, target_width):
    """Return height of page when scaled to given width"""
    d = fitz.open(pdf)
    w, h = d[0].rect.width, d[0].rect.height
    scale = target_width / w
    d.close()
    return h * scale

def place(page, pdf, x, y, target_width):
    """Place a PDF page on canvas at scaled width"""
    d = fitz.open(pdf)
    w, h = d[0].rect.width, d[0].rect.height
    scale = target_width / w
    r = fitz.Rect(x, y, x + target_width, y + h*scale)
    page.show_pdf_page(r, d, 0)
    d.close()
    return h*scale


# ================== 2×1 (STACKED) BUILD ==================
PAGE_W = 2000      # figure width
MARGIN = 60
GAP    = 100       # vertical space between panels

panel_width = PAGE_W - 2*MARGIN   # full width across both rows

h_top    = scaled_height(TOP_PANEL, panel_width)
h_bottom = scaled_height(BOTTOM_PANEL, panel_width)

PAGE_H = h_top + GAP + h_bottom + 2*MARGIN


# ================== CANVAS ==================
doc  = fitz.open()
page = doc.new_page(width=PAGE_W, height=PAGE_H)

# coords
y_T = MARGIN
y_B = MARGIN + h_top + GAP

# place PDFs
place(page, TOP_PANEL,    MARGIN, y_T, panel_width)
place(page, BOTTOM_PANEL, MARGIN, y_B, panel_width)


# ================== panel labels ==================
def label(letter, x, y):
    page.insert_text((x,y), letter, fontsize=48, fontname="helv", color=(0,0,0))

label("A", MARGIN, y_T - 30)          # above top panel
label("B", MARGIN, y_B - 30)          # above bottom panel


# ================== SAVE ==================
doc.save(OUT_PDF)
doc.close()
print("\nPDF saved →", OUT_PDF)

# PNG EXPORT 600 dpi
pdf = fitz.open(OUT_PDF)
pix = pdf[0].get_pixmap(dpi=600)
Image.frombytes("RGB",(pix.width,pix.height),pix.samples).save(OUT_PNG)
pdf.close()

print("PNG saved →", OUT_PNG)



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

PDF saved → Ablation_COMPARE_ORIGINAL_vs_CF_2x1.pdf
PNG saved → Ablation_COMPARE_ORIGINAL_vs_CF_2x1_600dpi.png


In [4]:
!pip install -q pymupdf pillow

import fitz
from PIL import Image

# ========== INPUT PDFs ==========
p_left  = "internet_usage_aggregate_regression_clean.pdf"   # Panel A
p_right = "gdp_aggregate_regression_clean.pdf"              # Panel B

# ========== OUTPUT ==========
out_pdf = "Figure3_1x2_VECTOR.pdf"
out_png = "Figure3_1x2_VECTOR_600dpi.png"

# ========== PAGE SETTINGS ==========
PAGE_W  = 1800
PAGE_H  = 900
MARGIN  = 60
GAP     = 80          # gap between left/right panels

# Width per panel
CELL_W = (PAGE_W - 2*MARGIN - GAP) / 2

def scaled_height(pdf, width):
    d = fitz.open(pdf)
    r = d[0].rect
    scale = width / r.width
    h = r.height * scale
    d.close()
    return h

# Compute heights so both panels match vertically
h_left  = scaled_height(p_left,  CELL_W)
h_right = scaled_height(p_right, CELL_W)
PANEL_H = max(h_left, h_right)

PAGE_H = MARGIN + PANEL_H + MARGIN

# Start page
doc = fitz.open()
page = doc.new_page(width=PAGE_W, height=PAGE_H)

# ========== PDF placement function with title-removal ==========
def place_clean(pdf_path, x, y, width):
    d = fitz.open(pdf_path)
    p = d[0]

    # Hide titles containing "(Aggregate - No Labels)"
    removal_text = "(Aggregate - No Labels)"
    for b in p.get_text("blocks"):
        if removal_text in b[4]:
            p.add_redact_annot(b[:4])
    p.apply_redactions()

    # Scale + place page
    r = p.rect
    scale = width / r.width
    h = r.height * scale
    dest = fitz.Rect(x, y, x + width, y + h)
    page.show_pdf_page(dest, d, 0)
    d.close()
    return h

# Coordinates
xL = MARGIN
xR = MARGIN + CELL_W + GAP
y  = MARGIN

# Place A / B
place_clean(p_left,  xL, y, CELL_W)
place_clean(p_right, xR, y, CELL_W)

# ===== Panel Labels (A / B) =====
def label(letter, x, y):
    page.insert_text((x, y), letter, fontsize=28, fontname="helv", color=(0,0,0))

label("A", xL, y - 30)
label("B", xR, y - 30)

# Save vector PDF
doc.save(out_pdf)
doc.close()
print("PDF saved:", out_pdf)

# Export 600-dpi PNG
pdf = fitz.open(out_pdf)
pix = pdf[0].get_pixmap(dpi=600)
Image.frombytes("RGB", (pix.width, pix.height), pix.samples).save(out_png)
pdf.close()
print("PNG saved:", out_png)



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
PDF saved: Figure3_1x2_VECTOR.pdf
PNG saved: Figure3_1x2_VECTOR_600dpi.png


In [5]:
"""
FIGURE 4 — Pluralistic Ignorance vs Claude Model Error
Israel removed as outlier. Median-based quadrant boundaries +
20 labelled countries for policy navigation.
"""

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ============================================================
# LOAD + PREP DATA
# ============================================================
pred = pd.read_csv("predictions_all_stages_long.csv")
gt   = pd.read_csv("data_final.csv")

pred8 = pred[pred['stage']==8].copy()

# True pluralistic ignorance = own − others
gt['pi_severity'] = gt['wtc_own'] - gt['wtc_other']
gt = gt[['countrynew','wtc_other','pi_severity']]

df = pred8.merge(gt, on='countrynew', how='inner')

# ---- Remove outlier Israel ----
df = df[df.countrynew != "Israel"]

# Claude model uncertainty = |predicted others − true others|
df['model_uncertainty'] = (df['pred_claude'] - df['wtc_other']).abs()



# ============================================================
# CONTINENT MAP (FULL — restored correctly)
# ============================================================
continent_map = {
    **{c:'Africa' for c in ['Algeria','Benin','Botswana','Burkina Faso','Cameroon','Congo Brazzaville',
                            'Egypt','Gabon','Ghana','Guinea','Ivory Coast','Kenya','Madagascar','Malawi',
                            'Mali','Morocco','Mozambique','Namibia','Nigeria','Senegal','Sierra Leone',
                            'South Africa','Tanzania','Togo','Tunisia','Uganda','Zambia','Zimbabwe']},

    **{c:'Asia' for c in ['Afghanistan','Armenia','Bangladesh','Cambodia','China','Georgia','Hong Kong',
                          'India','Indonesia','Iran','Iraq','Japan','Jordan','Kazakhstan',
                          'Kyrgyzstan','Laos','Lebanon','Malaysia','Mongolia','Myanmar','Nepal','Pakistan',
                          'Philippines','Saudi Arabia','Singapore','South Korea','Sri Lanka','Taiwan',
                          'Tajikistan','Thailand','Turkey','United Arab Emirates','Uzbekistan','Vietnam']},
    # Israel removed intentionally

    **{c:'Europe' for c in ['Albania','Austria','Belgium','Bosnia Herzegovina','Bulgaria','Croatia','Cyprus',
                            'Czech Republic','Denmark','Estonia','Finland','France','Germany','Greece',
                            'Hungary','Iceland','Ireland','Italy','Kosovo','Latvia','Lithuania','Malta',
                            'Moldova','Netherlands','North Macedonia','Norway','Poland','Portugal','Romania',
                            'Russia','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland',
                            'Ukraine','United Kingdom']},

    **{c:'North America' for c in ['Canada','Costa Rica','Dominican Republic','El Salvador','Guatemala',
                                   'Hondoras','Jamaica','Mexico','Nicaragua','Panama','United States']},

    **{c:'South America' for c in ['Argentina','Bolivia','Brazil','Chile','Colombia','Ecuador','Paraguay',
                                   'Peru','Uruguay','Venezuela']},

    **{c:'Oceania' for c in ['Australia','New Zealand','Mauritius']}
}

df['continent'] = df['countrynew'].map(continent_map)



# ============================================================
# MEDIANS → QUADRANTS FOR POLICY
# ============================================================
med_PI  = df['pi_severity'].median()
med_MAE = df['model_uncertainty'].median()

top_right     = df[(df.pi_severity>med_PI)&(df.model_uncertainty>med_MAE)].nlargest(5,'pi_severity')
top_left      = df[(df.pi_severity>med_PI)&(df.model_uncertainty<=med_MAE)].nlargest(5,'pi_severity')
bottom_right  = df[(df.pi_severity<=med_PI)&(df.model_uncertainty>med_MAE)].nlargest(5,'model_uncertainty')
bottom_left   = df[(df.pi_severity<=med_PI)&(df.model_uncertainty<=med_MAE)].nsmallest(5,'pi_severity')



# ============================================================
# PLOT
# ============================================================
fig, ax = plt.subplots(figsize=(11,8), dpi=300)

colors = {
    'Africa':'#e74c3c','Asia':'#f39c12','Europe':'#3498db',
    'North America':'#2ecc71','South America':'#9b59b6','Oceania':'#1abc9c'
}

# Scatter all countries
for cont in sorted(df['continent'].dropna().unique()):
    sub = df[df['continent']==cont]
    ax.scatter(sub['model_uncertainty'], sub['pi_severity'],
               s=85, color=colors[cont], alpha=0.78,
               edgecolors='white', linewidth=1, label=cont)




# LABEL 20 COUNTRIES
def label_block(block, dx, dy, bgcolor):
    for _, r in block.iterrows():
        ax.annotate(r['countrynew'],
            (r['model_uncertainty'], r['pi_severity']),
            fontsize=8, xytext=(dx,dy), textcoords='offset points',
            bbox=dict(boxstyle='round,pad=.3', fc=bgcolor, alpha=.78))

label_block(top_left,     dx=-6, dy=6,  bgcolor="lightyellow")  # high PI + low MAE
label_block(top_right,    dx=4,  dy=6,  bgcolor="gold")         # high PI + high MAE
label_block(bottom_left,  dx=-6, dy=-6, bgcolor="lightgreen")   # low PI + low MAE
label_block(bottom_right, dx=4,  dy=-6, bgcolor="lightblue")    # low PI + high MAE



# ============================================================
# QUADRANT LINES + POLICY TEXT
# ============================================================
ax.axhline(med_PI,  color="grey", lw=1.4, ls="--")
ax.axvline(med_MAE, color="grey", lw=1.4, ls="--")

ax.text(med_MAE*1.03, med_PI*1.03,
        "Survey-first priority\n(High PI × High Uncertainty)",
        fontsize=10, weight='bold')

ax.text(med_MAE*0.45, med_PI*1.03,
        "Intervention-ready\n(High PI × Low Uncertainty)",
        fontsize=10, weight='bold')

ax.text(med_MAE*1.03, med_PI*0.45,
        "Monitor + verify\n(Low PI × High Uncertainty)",
        fontsize=10, weight='bold')

ax.text(med_MAE*0.45, med_PI*0.45,
        "Low-cost reassurance zone\n(Low PI × Low Uncertainty)",
        fontsize=10, weight='bold')



# ============================================================
# TITLES + SAVE
# ============================================================
ax.set_xlabel("Claude Model Error (MAE — predicted others’ willingness)", fontsize=12.5)
ax.set_ylabel("Pluralistic Ignorance Severity (Own − Others, pp)", fontsize=12.5)

ax.set_title("Pluralistic Ignorance × Claude Uncertainty\n(Median Quadrants; Israel Removed)",
             fontsize=15, weight='bold')

ax.legend(title="Continent", loc='lower right', framealpha=0.92)
ax.grid(alpha=.25, ls=':')

plt.tight_layout()
plt.savefig("figure4_policy_quadrants_no_israel.pdf", dpi=300)
plt.savefig("figure4_policy_quadrants_no_israel.png", dpi=300)
plt.close()

print("\n✓ FIGURE BUILT — Quadrants + Policy Ready.")



✓ FIGURE BUILT — Quadrants + Policy Ready.


In [6]:
!pip install -q pymupdf pillow

import fitz
from PIL import Image

# ========= INPUT FILES (REPLACED) =========
panel_top    = "structured_mae_heatmap.pdf"
panel_bottom = "cf_structured_4panel.pdf"

# ========= OUTPUTS =========
out_pdf = "Fig_Structured_AB_2x1_VECTOR.pdf"
out_png = "Fig_Structured_AB_2x1_600dpi.png"


# ========= HELPER FUNCTIONS =========
def scaled_h(pdf_path, target_w):
    d = fitz.open(pdf_path)
    r = d[0].rect
    s = target_w / r.width
    h = r.height * s
    d.close()
    return h

def place_pdf(page, pdf_path, x, y, target_w):
    d = fitz.open(pdf_path)
    r = d[0].rect
    s = target_w / r.width
    h = r.height * s
    dest = fitz.Rect(x, y, x + target_w, y + h)
    page.show_pdf_page(dest, d, 0)
    d.close()
    return h


# ========= PAGE BUILD (2×1) =========
PAGE_W = 1800
MARGIN = 60
GAP_V  = 80   # gap between top & bottom panels

inner_w = PAGE_W - 2*MARGIN

h_top    = scaled_h(panel_top, inner_w)
h_bottom = scaled_h(panel_bottom, inner_w)

PAGE_H = MARGIN + h_top + GAP_V + h_bottom + MARGIN

doc = fitz.open()
page = doc.new_page(width=PAGE_W, height=PAGE_H)

# ----- coordinates -----
y_top     = MARGIN
y_bottom  = MARGIN + h_top + GAP_V

# ----- draw -----
place_pdf(page, panel_top,    MARGIN, y_top,    inner_w)
place_pdf(page, panel_bottom, MARGIN, y_bottom, inner_w)


# ========= PANEL LABELS =========
def add_label(letter, x, y):
    page.insert_text((x, y), letter,
                     fontsize=30, fontname="helv", color=(0,0,0))

add_label("A", MARGIN, y_top - 40)
add_label("B", MARGIN, y_bottom - 40)


# ========= SAVE =========
doc.save(out_pdf)
doc.close()
print("Master PDF saved:", out_pdf)

# ===== PNG EXPORT =====
pdf = fitz.open(out_pdf)
pix = pdf[0].get_pixmap(dpi=600)
img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
img.save(out_png)
pdf.close()

print("PNG saved:", out_png)



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Master PDF saved: Fig_Structured_AB_2x1_VECTOR.pdf
PNG saved: Fig_Structured_AB_2x1_600dpi.png


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>